In [1]:
import pandas as pd
import numpy as np

In [2]:
orders = pd.read_csv("C:/Users/User/OneDrive/Desktop/DA Intern @ ApexPlanet/Task-3/Raw Data/olist_orders_dataset.csv")
order_items = pd.read_csv("C:/Users/User/OneDrive/Desktop/DA Intern @ ApexPlanet/Task-3/Raw Data/olist_order_items_dataset.csv")
order_payments = pd.read_csv("C:/Users/User/OneDrive/Desktop/DA Intern @ ApexPlanet/Task-3/Raw Data/olist_order_payments_dataset.csv")
customers = pd.read_csv("C:/Users/User/OneDrive/Desktop/DA Intern @ ApexPlanet/Task-3/Raw Data/olist_customers_dataset.csv")
products = pd.read_csv("C:/Users/User/OneDrive/Desktop/DA Intern @ ApexPlanet/Task-3/Raw Data/olist_products_dataset.csv")
reviews = pd.read_csv("C:/Users/User/OneDrive/Desktop/DA Intern @ ApexPlanet/Task-3/Raw Data/olist_order_reviews_dataset.csv")
sellers = pd.read_csv("C:/Users/User/OneDrive/Desktop/DA Intern @ ApexPlanet/Task-3/Raw Data/olist_sellers_dataset.csv")
geolocation = pd.read_csv("C:/Users/User/OneDrive/Desktop/DA Intern @ ApexPlanet/Task-3/Raw Data/olist_geolocation_dataset.csv")

In [5]:
df = orders.merge(order_items, on="order_id", how="left")

In [7]:
df = df.merge(order_payments, on="order_id", how="left")

In [8]:
df = df.merge(customers, on="customer_id", how="left")

In [9]:
df = df.merge(products, on="product_id", how="left")

In [10]:
df = df.merge(sellers, on="seller_id", how="left")

In [11]:
 df = df.merge(reviews, on="order_id", how="left")

In [12]:
df["order_purchase_timestamp"] = pd.to_datetime(df["order_purchase_timestamp"])
df["order_delivered_customer_date"] = pd.to_datetime(df["order_delivered_customer_date"])

df["Year"] = df["order_purchase_timestamp"].dt.year
df["Month"] = df["order_purchase_timestamp"].dt.month_name()
df["Day"] = df["order_purchase_timestamp"].dt.day

In [ ]:
// Define core KPI


In [ ]:
1.Total Revenue

In [14]:
df["Revenue"] = df["price"] + df["freight_value"]

total_revenue = df["Revenue"].sum()

In [ ]:
2.Total Orders

In [15]:
total_orders = df["order_id"].nunique()
 

In [ ]:
3.Average Order Value

In [16]:
AOV = total_revenue / total_orders

In [ ]:
4.Total Customers

In [17]:
total_customers = df["customer_id"].nunique()

In [ ]:
5.Average Review Score

In [18]:
avg_review = df["review_score"].mean()

In [ ]:
// Deep-Dive Analysis
Cohort analysis

In [21]:
 
df['order_purchase_timestamp'] = pd.to_datetime(df['order_purchase_timestamp'])

 
df['OrderMonth'] = df['order_purchase_timestamp'].dt.to_period('M')

 
df['CohortMonth'] = df.groupby('customer_id')['OrderMonth'].transform('min')

 
df['CohortIndex'] = (
    (df['OrderMonth'].dt.year - df['CohortMonth'].dt.year) * 12 +
    (df['OrderMonth'].dt.month - df['CohortMonth'].dt.month)
)

cohort_data = df.groupby(['CohortMonth','CohortIndex'])['customer_id'].nunique().reset_index()

cohort_data.head()

,CohortMonth,CohortIndex,customer_id
0,2016-09,0,4
1,2016-10,0,324
2,2016-12,0,1
3,2017-01,0,800
4,2017-02,0,1780


In [22]:
customer_ltv = df.groupby('customer_id')['Revenue'].sum().reset_index()

customer_ltv.columns = ['customer_id','Total_Revenue']

customer_ltv.sort_values(by='Total_Revenue',ascending=False).head(10)

,customer_id,Total_Revenue
8546,1617b1357756262bfa56ab541c47bc16,13664.08
60184,9af2372a1e49340278e7c1ef8d749f34,13281.71
86603,de832e8dbb1f588a47013e53feaa67cc,11111.40
38590,63b964e79dee32a3587651701a2b8dbf,10553.28
43009,6f241d5bbb142b6f764387c8c270645a,10055.22
56788,926b6a6fb8b6081e00b335edaf578d35,8389.52
91663,eb7a157e8da9c488cd4ddc48711f1097,8068.88
96899,f959b7bc834045511217e6410985963f,8030.46
81790,d1ea705f2fdd8f98eff86c2691652e60,7413.70
91985,ec5b2ba62e574342386871631fafd3fc,7274.88


In [23]:
def segment(x):
    if x < 100:
        return "Low Value"
    elif x < 500:
        return "Medium Value"
    else:
        return "High Value"

customer_ltv['Segment'] = customer_ltv['Total_Revenue'].apply(segment)

customer_ltv.head()

,customer_id,Total_Revenue,Segment
0,00012a2ce6f8dcda20d059ce98491703,114.74,Medium Value
1,000161a058600d5901f007fab4c27140,67.41,Low Value
2,0001fd6190edaaf884bcaf3d49edf079,195.42,Medium Value
3,0002414f95344307404f0ace7a26f1d5,179.35,Medium Value
4,000379cdec625522490c315e70c7a9fb,107.01,Medium Value


In [24]:
category_sales = df.groupby('product_category_name')['Revenue'].sum().sort_values(ascending=False)

category_sales.head(10)

product_category_name
beleza_saude              1491397.76
relogios_presentes        1358845.59
cama_mesa_banho           1327662.02
esporte_lazer             1205197.85
informatica_acessorios    1104362.03
moveis_decoracao           955367.22
utilidades_domesticas      823623.50
cool_stuff                 752702.21
automotivo                 714431.95
ferramentas_jardim         625387.31
Name: Revenue, dtype: float64

In [25]:
df['delivery_time'] = (
df['order_delivered_customer_date'] -
df['order_purchase_timestamp']
).dt.days

delivery_reviews = df.groupby('review_score')['delivery_time'].mean()

delivery_reviews

review_score
1.0    19.099992
2.0    15.381893
3.0    13.552435
4.0    11.778330
5.0    10.203253
Name: delivery_time, dtype: float64

In [26]:
payment_methods = df['payment_type'].value_counts()

payment_methods

payment_type
credit_card    87776
boleto         23190
voucher         6465
debit_card      1706
not_defined        3
Name: count, dtype: int64

In [27]:
state_sales = df.groupby('customer_state')['Revenue'].sum().sort_values(ascending=False)

state_sales.head(10)

customer_state
SP    6234533.82
RJ    2247128.32
MG    1928571.09
RS     934286.75
PR     832059.40
BA     650220.38
SC     632514.14
GO     368997.61
DF     367750.15
ES     336438.54
Name: Revenue, dtype: float64

In [30]:
customer_orders = df.groupby('customer_id')['order_id'].nunique()

repeat_rate = (customer_orders[customer_orders > 1].count() /
               customer_orders.count()) * 100

In [ ]:
// dashboard dataset

In [31]:
df['Revenue'] = df['price'] + df['freight_value']

In [32]:
df.to_csv( "C:/Users/User/OneDrive/Desktop/DA Intern @ ApexPlanet/Task-3/MasterDataset.csv",index=False)